<a href="https://colab.research.google.com/github/abhisheksvcs25-bms/Bio-Inspired-Systems/blob/YT-Hand-Gesture-Controller/Cuckoo_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

data = pd.DataFrame({
    "Age": [20, 22, 21, 45, 47, 46, 25, 48],
    "Salary": [20000, 22000, 21000, 60000, 65000, 62000, 25000, 63000],
    "Gender": ["M", "F", "F", "M", "M", "F", "M", "F"],
    "Job": ["Student", "Student", "Student", "Engineer",
            "Engineer", "Engineer", "Student", "Engineer"]
})

numeric = ["Age", "Salary"]
categorical = ["Gender", "Job"]

K = 2
NESTS = 10
GENERATIONS = 100
ABANDON_PROBABILITY = 0.25

original_data = data.copy()

scaler = MinMaxScaler()
X = data.copy()
X[numeric] = scaler.fit_transform(X[numeric])


def distance(row, center):
    numeric_distance = sum(
        (float(row[col]) - float(center[col])) ** 2
        for col in numeric
    )

    categorical_distance = sum(
        row[col] != center[col] for col in categorical
    )

    return numeric_distance + categorical_distance


def assign_clusters(centers):
    labels = []
    total_cost = 0.0

    for _, row in X.iterrows():
        distances = [
            distance(row, centers.iloc[j])
            for j in range(K)
        ]

        cluster = int(np.argmin(distances))
        labels.append(cluster)
        total_cost += distances[cluster]

    return total_cost, np.array(labels, dtype=int)


def create_centers():
    indices = np.random.choice(len(X), K, replace=False)

    return X.iloc[indices].copy().reset_index(drop=True)


def update_centers(labels, old_centers):
    centers = []

    for cluster in range(K):
        indices = np.where(labels == cluster)[0]
        members = X.iloc[indices]

        if len(members) == 0:
            center = old_centers.iloc[cluster].to_dict()
        else:
            center = {}

            for col in numeric:
                center[col] = float(members[col].mean())

            for col in categorical:
                center[col] = members[col].mode().iloc[0]

        centers.append(center)

    return pd.DataFrame(
        centers, columns=X.columns
    ).reset_index(drop=True)


def improve_solution(centers):
    centers = centers.copy().reset_index(drop=True)

    cost, labels = assign_clusters(centers)

    for _ in range(10):
        new_centers = update_centers(labels, centers)
        new_cost, new_labels = assign_clusters(new_centers)

        if new_cost <= cost:
            centers = new_centers
            cost = new_cost
            labels = new_labels
        else:
            break

    return centers, cost, labels


def local_search(nest, best):
    candidate = nest.copy().reset_index(drop=True)

    for i in range(K):
        for col in numeric:
            difference = float(best.loc[i, col]) - float(
                candidate.loc[i, col]
            )

            candidate.loc[i, col] += (
                np.random.uniform(-1, 1) * difference
            )

        for col in categorical:
            if np.random.rand() < 0.5:
                candidate.loc[i, col] = best.loc[i, col]

    candidate[numeric] = candidate[numeric].clip(0, 1)

    return candidate


def global_search(nest, other1, other2):
    candidate = nest.copy().reset_index(drop=True)

    for i in range(K):
        for col in numeric:
            difference = (
                float(other1.loc[i, col])
                - float(other2.loc[i, col])
            )

            candidate.loc[i, col] += (
                np.random.uniform(-1, 1) * difference
            )

        for col in categorical:
            candidate.loc[i, col] = np.random.choice([
                nest.loc[i, col],
                other1.loc[i, col],
                other2.loc[i, col]
            ])

    candidate[numeric] = candidate[numeric].clip(0, 1)

    return candidate


nests = []

for _ in range(NESTS):
    centers = create_centers()
    centers, cost, labels = improve_solution(centers)
    nests.append((centers, cost, labels))

nests.sort(key=lambda item: item[1])

best_centers = nests[0][0].copy()
best_cost = nests[0][1]
best_labels = nests[0][2].copy()

print("Starting Cuckoo Search Clustering...")

for generation in range(GENERATIONS):

    updated_nests = []

    for centers, cost, labels in nests:
        candidate = local_search(centers, best_centers)

        candidate, candidate_cost, candidate_labels = improve_solution(
            candidate
        )

        if candidate_cost < cost:
            updated_nests.append((
                candidate, candidate_cost, candidate_labels
            ))
        else:
            updated_nests.append((centers, cost, labels))

    nests = sorted(updated_nests, key=lambda item: item[1])

    best_centers = nests[0][0].copy()
    best_cost = nests[0][1]
    best_labels = nests[0][2].copy()

    for i in range(NESTS):
        if np.random.rand() < ABANDON_PROBABILITY:

            other_indices = np.random.choice(
                NESTS, size=2, replace=False
            )

            candidate = global_search(
                nests[i][0],
                nests[other_indices[0]][0],
                nests[other_indices[1]][0]
            )

            candidate, candidate_cost, candidate_labels = improve_solution(
                candidate
            )

            if candidate_cost < nests[i][1]:
                nests[i] = (
                    candidate, candidate_cost, candidate_labels
                )

    nests.sort(key=lambda item: item[1])

    best_centers = nests[0][0].copy()
    best_cost = nests[0][1]
    best_labels = nests[0][2].copy()

    if (generation + 1) % 20 == 0:
        print(
            f"Generation {generation + 1}: "
            f"Best Cost = {best_cost:.4f}"
        )

result = original_data.copy()
result["Cluster"] = best_labels + 1

print("\n========== FINAL RESULTS ==========")
print("Number of Clusters:", K)
print("Best Clustering Cost:", round(best_cost, 4))

display_centers = best_centers.copy()
display_centers[numeric] = scaler.inverse_transform(
    display_centers[numeric]
)

print("\nCluster Centers:")
print(display_centers.to_string(index=False))

print("\nFinal Clustering Results:")
print(result.sort_values("Cluster").to_string(index=False))

for cluster in range(1, K + 1):
    print(f"\nCluster {cluster}:")

    cluster_data = result[result["Cluster"] == cluster]

    if cluster_data.empty:
        print("No records assigned to this cluster.")
    else:
        print(cluster_data.to_string(index=False))

print("\nClustering completed successfully.")

Starting Cuckoo Search Clustering...
Generation 20: Best Cost = 4.0376
Generation 40: Best Cost = 4.0376
Generation 60: Best Cost = 4.0376
Generation 80: Best Cost = 4.0376
Generation 100: Best Cost = 4.0376

========== FINAL RESULTS ==========
Number of Clusters: 2
Best Clustering Cost: 4.0376

Cluster Centers:
 Age  Salary Gender      Job
22.0 22000.0      F  Student
46.5 62500.0      F Engineer

Final Clustering Results:
 Age  Salary Gender      Job  Cluster
  20   20000      M  Student        1
  22   22000      F  Student        1
  21   21000      F  Student        1
  25   25000      M  Student        1
  45   60000      M Engineer        2
  47   65000      M Engineer        2
  46   62000      F Engineer        2
  48   63000      F Engineer        2

Cluster 1:
 Age  Salary Gender     Job  Cluster
  20   20000      M Student        1
  22   22000      F Student        1
  21   21000      F Student        1
  25   25000      M Student        1

Cluster 2:
 Age  Salary Gender  